 SDXL_Research
Caption, train, compare and generate SDXL LoRAs on the Colab free T4.

1. **Runtime > Change runtime type > T4 GPU**
2. **Secrets** (key icon, left): add `HF_TOKEN` (needed for private repos and sync);
   optional `CIVITAI_TOKEN` for Civitai downloads. Allow notebook access.
3. Run **Setup** (installs everything; re-runs skip finished steps), then **Launch**.

In [ ]:
#@title Setup: repo + venv (idempotent)
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/kopeng-mi/SDXL_Research.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO = Path("/content/SDXL_Research")

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "pull", "-q", "--ff-only"], cwd=REPO, check=True)
    subprocess.run(["git", "submodule", "update", "-q", "--init", "--recursive"], cwd=REPO, check=True)

# The kernel only orchestrates (stdlib); GPU work runs in venv subprocesses,
# so this kernel never holds VRAM.
sys.path.insert(0, str(REPO / "src"))
from sdxlr import nb
r = subprocess.run([sys.executable, "-m", "sdxlr.bootstrap"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                   env={**os.environ, "PYTHONPATH": str(REPO / "src")})
print(r.stdout[-3000:])
assert r.returncode == 0, "setup failed (see above)"
found = nb.load_secrets()
print("secrets:", ", ".join(f"{k} {'found' if v else 'missing'}" for k, v in found.items()))

# Launch
Opens the app. Everything else (datasets, captions, training, compare,
generation, downloads, presets) happens in the app. The cell keeps running
while the app is open; stop it to close the app.

In [ ]:
#@title Launch the app
PRIVATE_LINK = True  #@param {type:"boolean"}
#@markdown on: Colab's private link (only you). off: public gradio.live link (use if the private link does not open).
PULL_FROM_SYNC_REPO = True  #@param {type:"boolean"}
#@markdown pulls your presets + LoRAs from the sync repo saved in Library → Sync

args = ["-m", "sdxlr.app", "--port", 7860]
if not PRIVATE_LINK:
    args += ["--share"]
if PULL_FROM_SYNC_REPO:
    args += ["--sync"]
if PRIVATE_LINK:
    try:
        from google.colab.output import eval_js
        print("Open the app (ready when the log below says 'Running on'):", eval_js("google.colab.kernel.proxyPort(7860)"))
    except Exception:
        print("Not in Colab: open http://127.0.0.1:7860")
nb.run(args)